In [1]:
# Adding sources to the pythonpath
import sys
root_path = '../../../..'
sys.path.append(root_path)

import dotenv
dotenv.load_dotenv(dotenv.find_dotenv(root_path + '/.env'))

# Importing sources of our project
import src
import src.utils as utils

# Importing other libs
import json_tricks
import numpy
import random
import os

numpy.random.seed(0)
random.seed(0)


inputs1 = json_tricks.load(open('inputs/inputs1.json'))
inputs2 = json_tricks.load(open('inputs/inputs2.json'))

answer = {}


In [2]:
numpy.random.seed(0)
random.seed(0)


# Implementing a Backpropagation Package for NumPy

In this project, we will implement the backpropagation algorithm for the NumPy package.
Your task will be to implement all the methods of the `Node` class, which wraps NumPy arrays and supports backpropagation.

Step-by-step, we will implement the following methods:

1. `__init__` (a constructor)
2. `backward` (a recursive mechanism that triggers backpropagation)
3. `__neg__` (negation operator $- x$)
4. `__add__` (addition operator $x + y$)
5. `__mul__` (product operator $x \cdot y$)
6. `__sub__` (subtraction operator $x - y$)
7. `__truediv__` (division operator $x / y$)
8. `exp` (exponentiation $\exp(x)$)
9. `sum` (summation $\sum_{k} x_k$)
10. `matmul` (matrix product $XY$)

After that we will use the implemented methods on:
1. a simple graph from the previous task
2. a two-layer neural network

Let's start!

`Node([1]) + Node([2]) -> Node`
`(x + y).backward()`

# Task 1

Implement the `__init__` method for the `Node` class:
- create the field `data` and assign it to the `data` argument (we will need it to compute the gradient)
- create the field `grad` and assign it to `None` (it will be used to store the gradient of the node)
- create the field `inputs` and assign it to an empty list (it will be used to store the nodes on which the current node depends)
- create the field `n_dependents` and assign it to `0` (it will be used to count the number of nodes that depend on the current node)
- create the field `n_grads` and assign it to `0` (it will be used to count the number of gradients that come to the current node)

Implement the method `update_grad` for the `Node` class. The point of this method is to take care of the gradients that come from the next nodes and accumulate them:
- first, create a deep copy of `grad_output` (just in case): `grad_output = copy.deepcopy(grad_output)`
- increase the `n_grads` counter by 1 because one of the dependent gradients has been registered
- if `grad_output` is `None` and the size of `data` is 1, assign `grad_output` to the NumPy array `[1]` (as we can calculate derivatives only of scalar values without any dependencies)
- if `grad_output` is `None`, raise the `BaseException` with the message "Backpropagation is impossible" (as we can't compute the gradient without the gradient from the next node except for the case of a scalar value)
- if `self.grad` is `None`, assign `grad_output` to `self.grad` (that means that if there was no gradient before, we will just assign the one that we got from the next node)
- otherwise, add `grad_output` to `self.grad` (that means that if there was a gradient before, we will add the one that we got from the next node to the accumulated value)

Implement the `backward` method for the `Node` class. This method is the heart of the backpropagation algorithm. Its logic is the following:
- it waits and collects gradients from all dependents of the current node
- then it calculates the list of gradients for the inputs of the current node using the `backward_step` method (we will implement this method for every operation by hand)
- trigger the `backward` methods of the inputs with the calculated gradient in the current node as an argument

As a formula, it looks like this:
$\partial_w {L} = \sum_{k=1}^{n} \partial_w {z_k} \partial_{z_k} {L}$

So, how does this work in practice?
- call the `update_grad` method with the `grad_output` argument
- if we have collected all the gradients from the dependents, it is time to calculate the gradients for the inputs of the current node 
- call the `backward_step` method with the `grad_output` argument to calculate the gradients for the inputs
- call the `backward` method of the inputs with the calculated gradient in the current node as an argument
- reset the `n_grads` counter to 0 (actually, this is not necessary as we do backpropagation only once)

The `backward_step` method should also be implemented for a basic node too. It should return an empty list because a basic node does not have any inputs.

In general, we will store inputs to the operation as a list of nodes in the `inputs` field. The `backward_step` function should return the corresponding list of gradients for the inputs.

After these steps are done, you can check that the first test passes successfully.

# Task 2

Implement the `__add__` method for the `Node` class and the class `TensorSum` that inherits from `Node`. 
This method lets us use the `+` operator for a pair of `Node` objects. After we perform `z = x + y`, the result of this operation is also a `Node` that corresponds to the sum of two nodes.

The engine for that operation is the `TensorSum` class. Here you need to:
- initialize the `Node` class with the `None` argument (empty envelope of the node) in the `__init__` method
- in the `__call__` method (this method is used when we perform `z = x + y`):
    - assign the `inputs` field of the current node to `[input1, input2]` to store the inputs
    - set `n_dependents` to the sum of `n_dependents` of the inputs (or simply to 2 because we have only two inputs)
    - calculate the value of the `data` field of the current node using the `self.inputs[0].data + self.inputs[1].data` formula
    - set the `grad` field of the current node to `None` as we don't have any gradient yet
    - return the `self` node (so that the result of the operation is the current node)
- in the `backward_step` method:
    - return the list of gradients for the inputs: `[grad_output, grad_output]` as $\partial_x {x + y} = 1$ and $\partial_y {x + y} = 1$


In `Node` class:
- implement the `__add__` method:
    - create a new `TensorSum` object with the sum of the `data` of the current node and the `other` node and return it (`return TensorSum()(self, other)`)

Now your `Node` class supports the `+` operator. You can check it by running test number 2. So you can calculate gradients of expressions like `x + y + x + x + y + y`. That is not much, but it is a good start.

# Task 3 and others

Implement the remaining operations by analogy:
- `__sub__` that will be used when we perform `z = x - y`
- `__mul__` that will be used when we perform `z = x * y`
- `__truediv__` that will be used when we perform `z = x / y`
- `__neg__` that will be used when we perform `z = -x`
- `exp` that will be used when we perform `z = numpy.exp(x)`
- `sum` that will be used when we perform `z = numpy.sum(x)`
- `matmul` that will be used when we perform `z = x @ y`

In all these cases, use theoretical derivatives to implement the `backward_step` function for each of these operations.

The hardest one will be `matmul` as it is a matrix multiplication. To implement it, use the following rule of thumb:
- the resulting gradient should have the same shape as the data
- the gradients of linear functions are also linear functions

In [3]:
import numpy
import copy

# Node(np.array([1]))
class Node:
    def __init__(self, data):
        if data is not None and not isinstance(data, numpy.ndarray):
            self.data = numpy.array(data)
        else:
            self.data = data
        self.grad = None
        self.inputs = []
        self.n_dependents = 0
        self.n_grads = 0

    def update_grad(self, grad_output):
        grad_output = copy.deepcopy(grad_output)
        self.n_grads += 1
        if grad_output is None and self.data.size == 1:
            grad_output = numpy.array([1])
        elif grad_output is None:
            raise BaseException("Backpropagation is impossible")
        
        if self.grad is None:
            self.grad = grad_output
        else:
            self.grad += grad_output

    def backward(self, grad_output=None):
        self.update_grad(grad_output)
        if self.n_grads == max(1, self.n_dependents):
            grads = self.backward_step(self.grad)
            for inp, g in zip(self.inputs, grads):
                inp.backward(g)
            self.n_grads = 0

    def backward_step(self, grad_output=None):
        return []

    # @backprop
    # def backward(self, grad_output=None):


    def __str__(self):
        return f"Node(data={self.data}, grad={self.grad})"

    def __neg__(self):
        return TensorNeg()(self)

    def __add__(self, other):
        return TensorSum()(self, other)

    def __mul__(self, other):
        return TensorMul()(self, other)

    def __sub__(self, other):
        return TensorSub()(self, other)

    def __truediv__(self, other):
        return TensorDiv()(self, other)

    def exp(self):
        return TensorExp()(self)

    def sum(self, axis=None):
        return TensorSumReduce()(self)

    def __matmul__(self, other):
        return TensorMatMul()(self, other)

    def matmul(self, other):
        return TensorMatMul()(self, other)


class TensorSum(Node):
    def __init__(self):
        super().__init__(None)

    def __call__(self, input1, input2):
        self.inputs = [input1, input2]
        input1.n_dependents += 1
        input2.n_dependents += 1
        self.data = input1.data + input2.data
        self.grad = None
        return self

    def backward_step(self, grad_output=None):
        return [grad_output, grad_output]


class TensorSub(Node):
    def __init__(self):
        super().__init__(None)

    def __call__(self, input1, input2):
        self.inputs = [input1, input2]
        input1.n_dependents += 1
        input2.n_dependents += 1
        self.data = input1.data - input2.data
        self.grad = None
        return self

    def backward_step(self, grad_output=None):
        return [grad_output, -grad_output]


class TensorMul(Node):
    def __init__(self):
        super().__init__(None)

    def __call__(self, input1, input2):
        self.inputs = [input1, input2]
        input1.n_dependents += 1
        input2.n_dependents += 1
        self.data = input1.data * input2.data
        self.grad = None
        return self

    def backward_step(self, grad_output=None):
        return [grad_output * self.inputs[1].data,
                grad_output * self.inputs[0].data]


class TensorDiv(Node):
    def __init__(self):
        super().__init__(None)

    def __call__(self, input1, input2):
        self.inputs = [input1, input2]
        input1.n_dependents += 1
        input2.n_dependents += 1
        self.data = input1.data / input2.data
        self.grad = None
        return self

    def backward_step(self, grad_output=None):
        x = self.inputs[0].data
        y = self.inputs[1].data
        return [grad_output / y,
                -grad_output * x / y**2]


class TensorNeg(Node):
    def __init__(self):
        super().__init__(None)

    def __call__(self, input1):
        self.inputs = [input1]
        input1.n_dependents += 1
        self.data = -input1.data
        self.grad = None
        return self

    def backward_step(self, grad_output=None):
        return [-grad_output]


class TensorExp(Node):
    def __init__(self):
        super().__init__(None)

    def __call__(self, input1):
        self.inputs = [input1]
        input1.n_dependents += 1
        self.data = numpy.exp(input1.data)
        self.grad = None
        return self

    def backward_step(self, grad_output=None):
        return [grad_output * self.data]


class TensorSumReduce(Node):
    def __init__(self):
        super().__init__(None)

    def __call__(self, input1):
        self.inputs = [input1]
        input1.n_dependents += 1
        self.data = numpy.sum(input1.data)
        self.grad = None
        return self

    def backward_step(self, grad_output=None):
        return [numpy.ones_like(self.inputs[0].data) * grad_output]


class TensorMatMul(Node):
    def __init__(self):
        super().__init__(None)

    def __call__(self, input1, input2):
        self.inputs = [input1, input2]
        input1.n_dependents += 1
        input2.n_dependents += 1
        self.data = input1.data @ input2.data
        self.grad = None
        return self

    def backward_step(self, grad_output=None):
        A = self.inputs[0].data
        x = self.inputs[1].data
        if x.ndim == 1:
            return [numpy.outer(grad_output, x), A.T @ grad_output]
        return [grad_output @ x.T, A.T @ grad_output]

In [4]:
# TEST 1

x = Node(numpy.array([1]))
print(x)
x.backward()
print(x)

answer['init'] = []
for inp in inputs1:
    x = Node(inp['x'][0])

    x.backward()

    answer['init'].append(x.grad)


Node(data=[1], grad=None)
Node(data=[1], grad=[1])


In [5]:
# TEST 2

x = Node(numpy.array([1]))
y = Node(numpy.array([2]))

z = x + y + x + x + x + y

print(z)
z.backward()
print(x, y)

answer['sum'] = []
for inp in inputs1:
    x = Node(inp['x'][0])
    y = Node(inp['x'][1])

    z = x + y + x + x + x + y
    z.backward()

    answer['sum'].append(x.grad)


Node(data=[8], grad=None)
Node(data=[1], grad=[4]) Node(data=[2], grad=[2])


In [6]:
# TEST 3

x = Node(numpy.array([1]))
y = Node(numpy.array([2]))

z = x - y + x + x - x - y

print(z)
z.backward()
print(x, y)

answer['diff'] = []
for inp in inputs1:
    x = Node(inp['x'][0])
    y = Node(inp['x'][1])

    z = x - y + x + x - y - y
    z.backward()

    answer['diff'].append(x.grad)


Node(data=[-2], grad=None)
Node(data=[1], grad=[2]) Node(data=[2], grad=[-2])


In [7]:
# TEST 4

x = Node(numpy.array([1]))
y = Node(numpy.array([2]))

z = (x + y) * (x - y)

print(z)
z.backward()
print(x, y)

answer['mul'] = []
for inp in inputs1:
    x = Node(inp['x'][0])
    y = Node(inp['x'][1])

    z = (x + y) * (x - y) * (x + x + y)
    z.backward()
    answer['mul'].append(x.grad)


Node(data=[-3], grad=None)
Node(data=[1], grad=[2]) Node(data=[2], grad=[-4])


In [8]:
print()
print(answer['mul'])



[array([-6]), array([-174]), array([526]), array([34]), array([50]), array([258]), array([618]), array([-106]), array([158]), array([6]), array([318]), array([-58]), array([72]), array([-24]), array([202]), array([50]), array([34]), array([58]), array([384]), array([98]), array([216]), array([522]), array([96]), array([278]), array([-174]), array([-184]), array([54]), array([162]), array([18]), array([158]), array([54]), array([54]), array([24]), array([78]), array([58]), array([408]), array([54]), array([526]), array([414]), array([34]), array([-2]), array([54]), array([254]), array([102]), array([-34]), array([-78]), array([202]), array([442]), array([-78]), array([202]), array([50]), array([234]), array([262]), array([-106]), array([216]), array([18]), array([18]), array([-58]), array([-54]), array([78]), array([278]), array([34]), array([216]), array([-34]), array([-72]), array([-106]), array([642]), array([278]), array([-54]), array([18]), array([-162]), array([414]), array([632]

In [9]:
# TEST 5

x = Node(numpy.array([1]))
y = Node(numpy.array([2]))

z = x / y

print(z)
z.backward()
print(x, y)

answer['div'] = []
for inp in inputs1:
    x = Node(inp['x'][0])
    y = Node(inp['x'][1])

    z = x / (Node(0.5) + y)
    z.backward()

    answer['div'].append(x.grad)


Node(data=[0.5], grad=None)
Node(data=[1], grad=[0.5]) Node(data=[2], grad=[-0.25])


In [10]:
# TEST 6
with src.utils.safecatch():
    x = Node(numpy.array([1]))
    y = Node(numpy.array([2]))

    z = -x

    print(z)
    z.backward()
    print(x)

    answer['neg'] = []
    for inp in inputs1:
        x = Node(inp['x'][0])

        z = -x
        z.backward()

        answer['neg'].append(x.grad)


Node(data=[-1], grad=None)
Node(data=[1], grad=[-1])


In [11]:
# TEST 7

x = Node(numpy.array([1]))
y = Node(numpy.array([2]))

z = (x + y).exp()

print(z)
print(x.grad, y.grad)

z.backward()

print(x.grad, y.grad)

answer['exp'] = []
for inp in inputs1:
    x = Node(inp['x'][0])

    z = x.exp()
    z.backward()

    answer['exp'].append(x.grad)


Node(data=[20.08553692], grad=None)
None None
[20.08553692] [20.08553692]


# Task

Implement the graph function from the previous task. For constants, please use `Node(1)` whenever needed (otherwise you will get an error)

In [12]:
# TEST 8 (Graph)

def sigmoid(z):
    return Node(1) / (Node(1) + (-z).exp())

def tanh(z):
    return Node(2) * sigmoid(Node(2) * z) - Node(1)

def graph_value(x, w):
    b1, b2, c1, c2 = w
    
    z1 = x[0] + b1
    z2 = x[1] + b2
    z3 = sigmoid(z1)
    z4 = sigmoid(z2)
    z5 = tanh(z2)
    z6 = z5 * c2
    z7 = z1 * z4
    z8 = z7 * c1
    z9 = z3 * z6
    y = z8 + z9

    return y

answer['graph'] = []
for inp in inputs1:

    x = inp['x']
    w = inp['w']

    x = [Node(float(val)) for val in x]
    w = [Node(float(val)) for val in w]

    y = graph_value(x, w)
    y.backward()

    answer['graph'].append([x[0].grad, x[1].grad, w[0].grad, w[1].grad, w[2].grad, w[3].grad])

print("OUR RESULTS:")
print(x[0].grad, x[1].grad, w[0].grad, w[1].grad, w[2].grad, w[3].grad)

def torch_graph_value(x, w):
    b1, b2, c1, c2 = w[0], w[1], w[2], w[3]
    
    z1 = x[0] + b1
    z2 = x[1] + b2
    z3 = torch.sigmoid(z1)
    z4 = torch.sigmoid(z2)
    z5 = torch.tanh(z2)
    z6 = z5 * c2
    z7 = z1 * z4
    z8 = z7 * c1
    z9 = z3 * z6
    y = z8 + z9
    
    return y

import torch
x = torch.tensor(inp['x'], requires_grad=True, dtype=float)
w = torch.tensor(inp['w'], requires_grad=True, dtype=float)

y = torch_graph_value(x, w)
y.backward()

print("TORCH RESULTS:")
print(x.grad, w.grad)

OUR RESULTS:
[2.22355446] [-2.93231404] [2.22355446] [-2.93231404] [-3.65529289] [0.00509724]
TORCH RESULTS:
tensor([ 2.2236, -2.9323], dtype=torch.float64) tensor([ 2.2236, -2.9323, -3.6553,  0.0051], dtype=torch.float64)


# 2-layer NN

Implement a two-layer neural network and compute its gradient using the `Node` class:

$\mathbf y = \sigma( W_2 \sigma(W_1 \mathbf x + \mathbf b_1) + \mathbf b_2)$

Return the sum of all values in $y * y$ as the loss function.

In [13]:
# TEST 9 (Two-layer net)


def sigmoid(z):
    one = Node(numpy.array(1.0))
    return one / (one + (-z).exp())

def two_layer_net(x, W1, W2, b1, b2):
    z1 = W1.matmul(x) + b1
    a1 = sigmoid(z1)
    
    z2 = W2.matmul(a1) + b2
    y = sigmoid(z2)
    
    y = y * y
    
    return y.sum()


answer['two_layer_net'] = []
for inp in inputs2:
    x = Node(inp['x'])
    W1 = Node(inp['W1'])
    W2 = Node(inp['W2'])
    b1 = Node(inp['b1'])
    b2 = Node(inp['b2'])

    h_hat = two_layer_net(x, W1, W2, b1, b2)
    h_hat.backward()

    answer['two_layer_net'].append([x.grad, W1.grad, W2.grad, b1.grad, b2.grad])


# Conclusion

You have implemented a backpropagation algorithm. This algorithm is similar to the one used in PyTorch. Note that you have implemented all the mechanics of it. Thus, it should no longer be a magic box: you know exactly how it works.

In [14]:
json_tricks.dump(answer, '.answer.json')


'{"init": [{"__ndarray__": [1], "dtype": "int64", "shape": [1]}, {"__ndarray__": [1], "dtype": "int64", "shape": [1]}, {"__ndarray__": [1], "dtype": "int64", "shape": [1]}, {"__ndarray__": [1], "dtype": "int64", "shape": [1]}, {"__ndarray__": [1], "dtype": "int64", "shape": [1]}, {"__ndarray__": [1], "dtype": "int64", "shape": [1]}, {"__ndarray__": [1], "dtype": "int64", "shape": [1]}, {"__ndarray__": [1], "dtype": "int64", "shape": [1]}, {"__ndarray__": [1], "dtype": "int64", "shape": [1]}, {"__ndarray__": [1], "dtype": "int64", "shape": [1]}, {"__ndarray__": [1], "dtype": "int64", "shape": [1]}, {"__ndarray__": [1], "dtype": "int64", "shape": [1]}, {"__ndarray__": [1], "dtype": "int64", "shape": [1]}, {"__ndarray__": [1], "dtype": "int64", "shape": [1]}, {"__ndarray__": [1], "dtype": "int64", "shape": [1]}, {"__ndarray__": [1], "dtype": "int64", "shape": [1]}, {"__ndarray__": [1], "dtype": "int64", "shape": [1]}, {"__ndarray__": [1], "dtype": "int64", "shape": [1]}, {"__ndarray__": [